# Scanpy Benchmark for spatial mutliomcis data integration on simulated dataset

Notebook benchmarks spatial mutliomcis data integration using Scanpy on simulated dataset.

## Loading

In [ ]:
import omicverse as ov
import anndata as ad
import pandas as pd
import scanpy as sc
import numpy as np
import os

## Scanpy Pipeline

In [ ]:
# Set the directory for the datasets and the output directory
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for i in range(1, 6):
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad data.")
    # Read the RNA dataset
    adata_rna = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_rna = adata_rna.raw.to_adata()
    adata_rna.obs['ground_truth'] = adata_rna.obs['cell_type']

    # Preprocess the RNA data
    sc.pp.highly_variable_genes(adata_rna, n_top_genes=3000)
    adata_rna = adata_rna[:, adata_rna.var['highly_variable'] == True]
    sc.tl.pca(adata_rna)
    sc.pp.neighbors(adata_rna)
    sc.tl.umap(adata_rna)

    # Perform clustering using PCA representation
    ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['X_pca'].shape[1],
                    use_rep='X_pca')
    ov.utils.cluster(adata_rna, use_rep='X_pca', method='leiden', resolution=0.6)

    # Plot the spatial clustering results
    sc.pl.spatial(adata_rna, color=['ground_truth', 'leiden'], spot_size=0.12, wspace=0.4)

    # Save the processed dataset
    output_path = f'{output_dir}/Simulated_Dataset_{i}/scanpy_rna.h5ad'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    adata_rna.write_h5ad(output_path, compression='gzip')

In [ ]:
!pip list